# RAV-Med: Step 6c, a claim-free variant (cross-passage consensus)
**Weakness addressed.** ERS needs a trusted claim. Without one (label-free claim) it is at chance on HealthContradict and inverted on answer-no questions. A retrieval system often has several passages for one question. Can the passages be checked **against each other**, with no claim and no label?

**Method (fixed before running).** For each question, take the pool of all documents the benchmark provides for it. Each document is represented by its most claim-similar 80-word window (found with the label-free claim, so no label is used). For a document *i* the **consensus score** is the mean agreement with up to `K_OTHERS = 20` randomly chosen other documents of the same question: agreement(i, j) = 0.5 x [(P_entail - P_contra)(j to i) + (P_entail - P_contra)(i to j)], with the same roberta-large-mnli.

**Scorers compared (pairwise accuracy, correct > incorrect, question-level cluster bootstrap):**
ERS with the label-free claim; consensus alone; ERS-free + 0.5 x consensus; ERS-free + 1.0 x consensus (both weights fixed in advance, **not tuned**).

**What to expect, honestly.** Consensus can only help when correct documents are the **majority** in a pool, because two disagreeing documents are symmetric: the method sees a conflict but cannot say which side is right. The notebook reports how often correct documents are the majority, and accuracy split by that. If consensus does not beat ERS-free, report it as a negative result: it still tells readers that peer-passage checking is not a cure for the missing claim.

**Uploads needed:** `hc_claims.csv`. Set `DRY_RUN = True` first.

In [ ]:
!pip install -q sentence-transformers transformers accelerate bitsandbytes scikit-learn scipy datasets

In [ ]:
import os, re, json, random, hashlib, subprocess, shutil, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED)
rng = np.random.default_rng(SEED)

DRY_RUN = False        # True = tiny fake models, no GPU: checks that every cell runs in seconds. Set False for real results.
DEVICE = "cuda"
EMBED_MODEL = "pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb"   # as in the paper
NLI_MODEL   = "roberta-large-mnli"                                        # as in the paper
N_BOOT = 2000
OUT = "step6_outputs"; os.makedirs(OUT, exist_ok=True)

# ---- platform helpers: the same notebook runs on Google Colab and on Kaggle ----
KAGGLE_ROOT = "/kaggle"
ON_KAGGLE = os.path.isdir(KAGGLE_ROOT + "/working")
def _find_in_kaggle_input(name):
    for root, _, fs in os.walk(KAGGLE_ROOT + "/input"):
        if name in fs: return os.path.join(root, name)
    return None
def ensure_file(name):
    """Make `name` available in the working directory (Kaggle: from an attached dataset; Colab: upload dialog)."""
    if os.path.exists(name): return
    if ON_KAGGLE:
        p = _find_in_kaggle_input(name)
        if p is None: raise FileNotFoundError(f"{name} not found. In Kaggle: Add Input -> upload it as a dataset and attach it to this notebook.")
        shutil.copy(p, name); print("copied", name, "from", p); return
    try:
        from google.colab import files; print("Upload", name); files.upload()
    except ImportError:
        raise FileNotFoundError(name)
def load_cache(path):
    """Read a JSON cache; if the file is truncated (disconnect during a write), fall back to the .bak copy."""
    for p in (path, path + ".bak"):
        if os.path.exists(p):
            try: return json.load(open(p))
            except Exception as e: print("cache file", p, "unreadable:", e)
    return {}
def save_cache(cache, path):
    """Write atomically (temp file, then rename) and keep one backup, so a disconnect cannot corrupt the cache."""
    tmp = path + ".tmp"; json.dump(cache, open(tmp, "w"))
    if os.path.exists(path): shutil.copy(path, path + ".bak")
    os.replace(tmp, path)
def restore_cache(path):
    """Kaggle only: if a cache file from an earlier session is attached as input, copy it back."""
    if ON_KAGGLE and not os.path.exists(path):
        p = _find_in_kaggle_input(os.path.basename(path))
        if p: shutil.copy(p, path); print("restored cache from", p)

K_OTHERS = 20
LAMBDAS = [0.5, 1.0]

In [ ]:
import torch
def _h(s): return int(hashlib.md5(s.encode("utf-8")).hexdigest()[:8], 16) / 0xffffffff

if DRY_RUN:
    def embed(texts, bs=64):
        out = []
        for t in texts:
            v = np.random.default_rng(int(_h(t) * 1e9)).normal(size=16); out.append(v / np.linalg.norm(v))
        return np.array(out)
    def nli3(prem, hyp, bs=32):
        rows = []
        for p, h in zip(prem, hyp):
            r = np.random.default_rng(int(_h(p[:200] + "||" + h) * 1e9)).random(3); rows.append(r / r.sum())
        return np.array(rows)          # columns: contradiction, neutral, entailment
else:
    from sentence_transformers import SentenceTransformer
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    embedder = SentenceTransformer(EMBED_MODEL, device=DEVICE)
    print("SBERT max_seq_length (as in all earlier runs):", embedder.max_seq_length)
    nli_tok = AutoTokenizer.from_pretrained(NLI_MODEL)
    nli = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL).to(DEVICE).eval()
    _lab = {l.lower(): i for i, l in nli.config.id2label.items()}
    _IDX = [_lab["contradiction"], _lab["neutral"], _lab["entailment"]]
    def embed(texts, bs=64):
        return embedder.encode(list(texts), batch_size=bs, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=len(texts) > 500)
    @torch.no_grad()
    def nli3(prem, hyp, bs=32):
        prem, hyp = list(prem), list(hyp)
        order = np.argsort([len(p) for p in prem]); out = np.zeros((len(prem), 3))
        for i in range(0, len(prem), bs):
            idx = order[i:i+bs]
            enc = nli_tok([prem[j] for j in idx], [hyp[j] for j in idx], truncation=True, padding=True, max_length=256, return_tensors="pt").to(DEVICE)
            out[idx] = torch.softmax(nli(**enc).logits.float(), -1)[:, _IDX].cpu().numpy()
        return out
def contra(prem, hyp, bs=32): return nli3(prem, hyp, bs)[:, 0]
def cos(a, b): return (embed(list(a)) * embed(list(b))).sum(1)

from sklearn.metrics import roc_auc_score
def auroc_ci(y, x, cluster, n_boot=N_BOOT, seed=SEED):
    """AUROC with a cluster bootstrap (resample whole clusters, e.g. queries or claims)."""
    y = np.asarray(y); x = np.asarray(x, float); cl = np.asarray(cluster)
    a = roc_auc_score(y, x); r = np.random.default_rng(seed); u = np.unique(cl); pos = {c: np.where(cl == c)[0] for c in u}; b = []
    for _ in range(n_boot):
        idx = np.concatenate([pos[c] for c in r.choice(u, len(u), replace=True)])
        if len(np.unique(y[idx])) == 2: b.append(roc_auc_score(y[idx], x[idx]))
    return a, np.percentile(b, 2.5), np.percentile(b, 97.5)

def win(a, b): return (np.asarray(a) > np.asarray(b)).astype(float) + 0.5 * (np.asarray(a) == np.asarray(b))
def pairwise_ci(w, topic, n_boot=N_BOOT, seed=SEED):
    """Pairwise accuracy (pooled over pairs) with a question-level cluster bootstrap, as in the HealthContradict step."""
    w = np.asarray(w, float); topic = np.asarray(topic); u = np.unique(topic); r = np.random.default_rng(seed)
    s = np.array([w[topic == t].sum() for t in u]); n = np.array([(topic == t).sum() for t in u]); bi = r.integers(0, len(u), size=(n_boot, len(u)))
    b = s[bi].sum(1) / n[bi].sum(1)
    return w.mean(), np.percentile(b, 2.5), np.percentile(b, 97.5)

In [ ]:
import hashlib
HC_JSONL = os.environ.get("HC_JSONL", "HealthContradict/dataset/dataset_ready.jsonl")
if not os.path.exists(HC_JSONL):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/tinaboya/HealthContradict"], check=True)
raw = pd.DataFrame([json.loads(l) for l in open(HC_JSONL, encoding="utf-8")]); raw["topic_id"] = raw["topic_id"].astype(int)
raw["correct_text"]   = np.where(raw.query_stance == "yes", raw.doc_a, raw.doc_b)     # doc_a always has stance yes, doc_b no
raw["incorrect_text"] = np.where(raw.query_stance == "yes", raw.doc_b, raw.doc_a)
did = lambda t: hashlib.md5(t.encode("utf-8")).hexdigest()[:12]
raw["correct_id"] = raw.correct_text.map(did); raw["incorrect_id"] = raw.incorrect_text.map(did)
docs = {}
for col in ["correct", "incorrect"]:
    for i, t in zip(raw[f"{col}_id"], raw[f"{col}_text"]): docs[i] = t
pairs = raw[["instance_id", "topic_id", "query", "query_stance", "correct_id", "incorrect_id"]].copy()
topics = pairs.drop_duplicates("topic_id").set_index("topic_id")[["query", "query_stance"]]
print(f"{len(pairs)} pairs | {len(topics)} questions | {len(docs)} unique documents")

# claims written by the earlier step (upload hc_claims.csv from the repository: data/healthcontradict/hc_claims.csv)
ensure_file("hc_claims.csv")
claims = pd.read_csv("hc_claims.csv").set_index("topic_id")
assert set(claims.index) == set(topics.index), "hc_claims.csv does not match the 81 questions"
CHUNK_WORDS, MIN_CHUNK_WORDS, MAX_CHUNKS = 80, 15, 120
def chunk(text):
    w = text.split(); ch = [" ".join(w[i:i+CHUNK_WORDS]) for i in range(0, len(w), CHUNK_WORDS)]
    if len(ch) > 1 and len(ch[-1].split()) < MIN_CHUNK_WORDS: ch = ch[:-1]
    return ch[:MAX_CHUNKS] if ch else [text]
need = pd.concat([pairs[["topic_id"]].assign(doc_id=pairs.correct_id), pairs[["topic_id"]].assign(doc_id=pairs.incorrect_id)]).drop_duplicates().reset_index(drop=True)
doc_ids = sorted(need.doc_id.unique())
chunks = pd.DataFrame([(d, k, c) for d in doc_ids for k, c in enumerate(chunk(docs[d]))], columns=["doc_id", "k", "text"])
chunk_emb = embed(chunks.text.tolist())
chunk_index = chunks.reset_index().groupby("doc_id")["index"].apply(list).to_dict()
print(f"{len(doc_ids)} documents -> {len(chunks)} windows")

def top_windows(claim_type):
    """The most claim-similar 80-word window of every (question, document): what a retriever would hand to a generator."""
    rows = []
    for t, d in need.itertuples(index=False):
        ce = embed([claims.loc[t, f"claim_{claim_type}"]])[0]
        ids = chunk_index[d]; S = chunk_emb[ids] @ ce; j = int(np.argmax(S))
        rows.append(dict(topic_id=t, doc_id=d, claim_type=claim_type, window_index=j, S=float(S[j]), window=chunks.text[ids[j]], claim=claims.loc[t, f"claim_{claim_type}"]))
    return pd.DataFrame(rows)

In [ ]:
tf = top_windows("free"); tf["C"] = contra(tf.window.tolist(), tf.claim.tolist()); tf["ERS_free"] = tf.S - tf.C
W = tf.set_index(["topic_id", "doc_id"])
r3 = np.random.default_rng(SEED)
pairs_ij = []                                         # (topic, i, j) sampled neighbours
for t, g in need.groupby("topic_id"):
    ds = sorted(g.doc_id.unique())
    for i in ds:
        others = [d for d in ds if d != i]
        if len(others) > K_OTHERS: others = list(r3.choice(others, K_OTHERS, replace=False))
        pairs_ij += [(t, i, j) for j in others]
P = pd.DataFrame(pairs_ij, columns=["topic_id", "i", "j"]); print(len(P), "neighbour pairs")
wi = [W.loc[(t, i), "window"] for t, i in zip(P.topic_id, P.i)]; wj = [W.loc[(t, j), "window"] for t, j in zip(P.topic_id, P.j)]
a_ji = nli3(wj, wi); a_ij = nli3(wi, wj)                # premise j -> hypothesis i, and reverse
P["agree"] = 0.5 * ((a_ji[:, 2] - a_ji[:, 0]) + (a_ij[:, 2] - a_ij[:, 0]))
P["conflict"] = 0.5 * (a_ji[:, 0] + a_ij[:, 0])
cons = P.groupby(["topic_id", "i"]).agg(CONS=("agree", "mean"), CONFLICT=("conflict", "mean"), n_nb=("agree", "size")).reset_index().rename(columns={"i": "doc_id"})
tf = tf.merge(cons, on=["topic_id", "doc_id"], how="left"); tf["CONS"] = tf.CONS.fillna(0.0)
for L_ in LAMBDAS: tf[f"ERS_free+{L_}CONS"] = tf.ERS_free + L_ * tf.CONS
tf.drop(columns=["window"]).to_csv(f"{OUT}/s6c_consensus_scores_nowindowtext.csv", index=False)
print(tf[["S", "C", "ERS_free", "CONS", "CONFLICT"]].describe().round(3))

## Pairwise accuracy (correct document vs incorrect document of the same pair)

In [ ]:
K = tf.set_index(["topic_id", "doc_id"])
SCN = {"Similarity only, label-free claim": ("S", 1), "ERS, label-free claim": ("ERS_free", 1), "Consensus only (claim-free)": ("CONS", 1),
       **{f"ERS-free + {l} x consensus": (f"ERS_free+{l}CONS", 1) for l in LAMBDAS}}
def pair_vals(col):
    c_ = K.loc[list(zip(pairs.topic_id, pairs.correct_id)), col].values; i_ = K.loc[list(zip(pairs.topic_id, pairs.incorrect_id)), col].values
    return c_, i_
rows = []
for sn, (col, sg) in SCN.items():
    c_, i_ = pair_vals(col); w = win(sg * c_, sg * i_)
    m, lo, hi = pairwise_ci(w, pairs.topic_id.values); row = dict(scorer=sn, scope="all pairs", n_pairs=len(w), pairwise_acc=m, lo=lo, hi=hi); rows.append(row)
    for stance in ["yes", "no"]:
        mk = (pairs.query_stance == stance).values; m, lo, hi = pairwise_ci(w[mk], pairs.topic_id.values[mk])
        rows.append(dict(scorer=sn, scope=f"answer-{stance} questions", n_pairs=int(mk.sum()), pairwise_acc=m, lo=lo, hi=hi))
# is the correct side the majority of each question's pool?
maj = pd.DataFrame({"correct_docs": pairs.groupby("topic_id").correct_id.nunique(), "incorrect_docs": pairs.groupby("topic_id").incorrect_id.nunique()})
maj["correct_majority"] = maj.correct_docs > maj.incorrect_docs; maj["tie"] = maj.correct_docs == maj.incorrect_docs
print(f"Questions where correct documents are the majority of the pool: {maj.correct_majority.mean():.1%} (ties {maj.tie.mean():.1%}) of {len(maj)}")
cm = pairs.topic_id.map(maj.correct_majority).values.astype(bool)
for sn, (col, sg) in SCN.items():
    c_, i_ = pair_vals(col); w = win(sg * c_, sg * i_)
    for nm, mk in [("pairs in correct-majority pools", cm), ("pairs in other pools", ~cm)]:
        if mk.sum() > 0 and len(np.unique(pairs.topic_id.values[mk])) > 1:
            m, lo, hi = pairwise_ci(w[mk], pairs.topic_id.values[mk]); rows.append(dict(scorer=sn, scope=nm, n_pairs=int(mk.sum()), pairwise_acc=m, lo=lo, hi=hi))
T = pd.DataFrame(rows); print(T.round(3).to_string(index=False)); T.to_csv(f"{OUT}/s6c_consensus_pairwise.csv", index=False); maj.to_csv(f"{OUT}/s6c_pool_majority.csv")

## Part B: controlled-majority pools (the informative test)
In HealthContradict the pools are balanced by construction (every pair adds one correct and one incorrect document), so Part A cannot show whether peer checking works. Here we **build** pools of 5 documents from one question: *4 correct + 1 incorrect* (the assumption behind any consensus method: most sources are right) and the reverse *1 correct + 4 incorrect* (consensus should then fail, and would drop the correct document). Success = the single minority document gets the **lowest** score in its pool (chance = 20%). For comparison, ERS with the label-free claim is scored the same way. Pools are drawn from at most 15 correct + 15 incorrect documents per question; 200 pools per question and scenario; question-level bootstrap.

In [ ]:
POOL, R_POOLS, CAP = 5, 200, 15
r4 = np.random.default_rng(SEED)
cor_t = pairs.groupby("topic_id").correct_id.apply(set); inc_t = pairs.groupby("topic_id").incorrect_id.apply(set)
print("documents listed as both correct and incorrect within a question (removed):", sum(len(cor_t[t] & inc_t[t]) for t in cor_t.index))
ERSF = tf.set_index(["topic_id", "doc_id"]).ERS_free.to_dict()
res = {"4 correct + 1 incorrect": [], "1 correct + 4 incorrect": []}
for t in sorted(cor_t.index):
    Cd = sorted(cor_t[t] - inc_t[t]); Id = sorted(inc_t[t] - cor_t[t])
    Cd = list(r4.choice(Cd, min(CAP, len(Cd)), replace=False)) if Cd else []; Id = list(r4.choice(Id, min(CAP, len(Id)), replace=False)) if Id else []
    ds = Cd + Id
    if len(Cd) + len(Id) < POOL: continue
    ix = {d: k for k, d in enumerate(ds)}; oi = [(i, j) for i in ds for j in ds if i != j]
    A = nli3([W.loc[(t, j), "window"] for i, j in oi], [W.loc[(t, i), "window"] for i, j in oi]); M = np.zeros((len(ds), len(ds)))
    for (i, j), a in zip(oi, A): M[ix[i], ix[j]] = a[2] - a[0]
    SYM = 0.5 * (M + M.T)
    for name, (nmaj, nmin, majset, minset) in {"4 correct + 1 incorrect": (POOL - 1, 1, Cd, Id), "1 correct + 4 incorrect": (POOL - 1, 1, Id, Cd)}.items():
        if len(majset) < nmaj or len(minset) < nmin: continue
        hit_c, hit_e = [], []
        for _ in range(R_POOLS):
            mem = list(r4.choice(majset, nmaj, replace=False)) + list(r4.choice(minset, nmin, replace=False)); k = [ix[d] for d in mem]
            cons_ = np.array([np.mean([SYM[a, b] for b in k if b != a]) for a in k]) + r4.normal(0, 1e-9, POOL)   # tiny noise = random tie-break
            ers_ = np.array([ERSF[(t, d)] for d in mem]) + r4.normal(0, 1e-9, POOL)
            hit_c.append(int(np.argmin(cons_) == POOL - 1)); hit_e.append(int(np.argmin(ers_) == POOL - 1))
        res[name].append((t, np.mean(hit_c), np.mean(hit_e)))
rows = []
for name, v in res.items():
    if not v: continue
    a = pd.DataFrame(v, columns=["topic", "cons", "ers"])
    for col, lab in [("cons", "Consensus (claim-free)"), ("ers", "ERS, label-free claim")]:
        x = a[col].values; b = [x[r4.integers(0, len(x), len(x))].mean() for _ in range(N_BOOT)]
        rows.append(dict(scenario=name, scorer=lab, n_questions=len(a), minority_doc_ranked_lowest=x.mean(), lo=np.percentile(b, 2.5), hi=np.percentile(b, 97.5), chance=1 / POOL))
T_POOL = pd.DataFrame(rows); print(T_POOL.round(3).to_string(index=False)); T_POOL.to_csv(f"{OUT}/s6c_controlled_majority_pools.csv", index=False)

In [ ]:
import shutil
shutil.make_archive("rav_med_step6c_outputs", "zip", OUT)
try:
    from google.colab import files; files.download("rav_med_step6c_outputs.zip")
except ImportError: pass
print("Saved rav_med_step6c_outputs.zip")